In [ ]:
!nvidia-smi

In [ ]:
#!pip install --upgrade langchain
#!pip install langchain_community
#!pip install langchain_text_splitter
#!pip install langchain_chroma
#!pip install langchain_groq
#!pip install langchain

In [ ]:
#!pip install unstructured
!pip install langchain-community "unstructured[pdf]"

In [ ]:
import os
from langchain_community.document_loaders import UnstructuredFileLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA

In [ ]:
os.environ["GROQ_API_KEY"] = "YOUR_SECRET_KEY"

In [ ]:
import requests
url = "https://web.stanford.edu/~jurafsky/slp3/11.pdf"
response = requests.get(url)

In [ ]:
with open("rag.pdf", "wb") as f:
  f.write(response.content)

In [ ]:
loader = UnstructuredFileLoader("rag.pdf")

In [ ]:
loader

In [ ]:
documents = loader.load()
documents

In [ ]:
text_splitter = CharacterTextSplitter(
    chunk_size = 3000,
    chunk_overlap = 400
)

In [ ]:
texts = text_splitter.split_documents(documents)

In [ ]:
type(texts)

In [ ]:
len(texts)

In [ ]:
texts[2]

In [ ]:
embeddings = HuggingFaceEmbeddings()

In [ ]:
persist_directory = "vector_db"

In [ ]:
vectordb = Chroma.from_documents(
    documents = texts,
    embedding=embeddings,
    persist_directory=persist_directory
)

In [ ]:
retriver = vectordb.as_retriever()

In [ ]:
llm = ChatGroq(
     model ="qwen/qwen3.8-27b",
     temperature=1
 )

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm =llm,
    chain_type = "stuff",
    retriever=retriver,
    return_source_documents = True
)

In [ ]:
query = "what do you mean by RAG?"
response = qa_chain.invoke({"query": query})

In [ ]:
print(response)

In [ ]:
import gradio as gr

def rag_chat(message, history):
    try:
        response = qa_chain.invoke({
            "query": message
        })

        answer = response["result"]
        source_documents = response.get("source_documents", [])

        sources = []
        for doc in source_documents:
            source = doc.metadata.get("source", "Unknown")
            page = doc.metadata.get("page", "N/A")
            sources.append(f"{source} (page {page})")

        # Duplicate sources hatao, order same rakho
        sources = list(dict.fromkeys(sources))

        if sources:
            sources_text = "\n".join(f"- {s}" for s in sources)
            return f"{answer}\n\n**Sources:**\n{sources_text}"
        return answer

    except Exception as e:
        return f"Error aaya: {str(e)}"


demo = gr.ChatInterface(
    fn=rag_chat,
    title="RAG PDF Chatbot",
    description="Ask RAG based Questions only!",
    examples=[
        "What do you mean by RAG?",
        "Give me a short summary of this pdf.",
    ],
)

demo.launch()